# 05 · Tuning HistGradientBoosting

Can better hyperparameters reduce HistGradientBoosting's overfitting (train/validation gap 0.022 in notebook 03) and raise its ROC-AUC?

Everything is measured with the same 5-fold CV as notebooks 03 and 04, on the data with the `weighted_past_due` feature from notebook 04, and compared with the default model **fold by fold**.

In [1]:
import numpy as np 
import pandas as pd
from scipy.stats import loguniform, randint
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_validate

from credit_risk_ai.data.clean import clean, drop_duplicate_rows
from credit_risk_ai.data.features import add_features
from credit_risk_ai.data.load import load_train
from credit_risk_ai.models.pipeline import build_hist_gradient_boosting, build_tuned_hist_gradient_boosting

N_SPLITS = 5
RANDOM_STATE = 42
N_ITER = 30
TARGET = "SeriousDlqin2yrs"

## Step 1 · Data

Cleaned, deduplicated training data plus `weighted_past_due` (the feature kept in notebook 04): 149,391 rows, 15 features.

In [2]:
data = add_features(clean(drop_duplicate_rows(load_train())))

X = data.drop(columns=[TARGET])
y = data[TARGET]

X.shape

(149391, 15)

## Step 2 · Baseline: the default model

Same folds as notebooks 03 and 04. `return_train_score=True` also scores each fold's model on the rows it was trained on, so the train/validation gap (overfitting) can be measured. The validation score must match notebook 04 (0.8651).

In [3]:
cv = StratifiedKFold(N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
baseline = cross_validate(build_hist_gradient_boosting(), X, y, cv=cv, scoring="roc_auc", return_train_score=True, n_jobs=-1)

val = baseline["test_score"].mean()
train = baseline["train_score"].mean()

print(f"validation: {val:.4f}")
print(f"train: {train:.4f}")
print(f"gap : {train - val:.4f}")

validation: 0.8651
train: 0.8874
gap : 0.0223


## Step 3 · Hyperparameters, search space and decision rule

| Hyperparameter | What it controls | Search range | Less overfitting when |
|---|---|---|---|
| `learning_rate` | How much each new tree corrects the previous ones | 0.01 – 0.3 (log scale) | smaller |
| `max_leaf_nodes` | How complex each tree can be | 8 – 63 | smaller |
| `min_samples_leaf` | Minimum number of rows in each leaf | 20 – 499 | larger |
| `l2_regularization` | Penalty on large leaf values | 0.001 – 10 (log scale) | larger |

`max_iter` is raised from 100 to 500 so that small learning rates have enough trees; early stopping (on by default with more than 10,000 rows) stops adding trees when they no longer help.

`RandomizedSearchCV` tries 30 random combinations instead of every combination (a grid of 5 values each would be 625): usually only one or two hyperparameters matter, and random sampling tries many different values of each. Log-scale ranges (`loguniform`) spread the tries evenly over orders of magnitude (0.01–0.03 gets as many tries as 0.1–0.3).

**Decision rule** (same as notebook 04, fixed before the search): adopt the tuned model if it beats the default on **at least 4 of 5 folds** AND the **mean gain is at least +0.001**.

## Step 4 · Random search

30 candidates × 5 folds = 150 fits (about 1 minute). `random_state` fixes which 30 combinations are tried, and the search uses the same `cv` object as the baseline. `refit=False`: only the scores are needed here; the final model is trained later.

In [4]:
model = build_hist_gradient_boosting()
model.set_params(max_iter=500)

param_distributions = {
    "learning_rate": loguniform(0.01, 0.3),
    "max_leaf_nodes": randint(8, 64),
    "min_samples_leaf": randint(20, 500),
    "l2_regularization": loguniform(0.001, 10)
}

search = RandomizedSearchCV(model, param_distributions, n_iter=N_ITER, scoring="roc_auc", cv=cv, random_state=RANDOM_STATE, n_jobs=-1, return_train_score=True, refit=False)

search.fit(X, y)
print(f"Best score: {search.best_score_:.4f}")
print("Best params:", {name: round(float(value), 4) for name, value in search.best_params_.items()})


Best score: 0.8663
Best params: {'l2_regularization': 1.1044, 'learning_rate': 0.0675, 'max_leaf_nodes': 21.0, 'min_samples_leaf': 378.0}


## Step 5 · Top 10 candidates

How to read the table below:

- **`learning_rate` matters most.** All of the top 10 use 0.013–0.068; the three worst candidates use 0.19–0.27.
- `min_samples_leaf` is mostly large (150–407) in the top 10, but not always (28 in one of them). `max_leaf_nodes` (10–30) and `l2_regularization` (0.0015–5.9, all over its range) matter little.
- **The top 10 are within 0.0005 of each other**, while each candidate's std across folds is about 0.004: a flat plateau of good settings, not one sharp optimum.
- **The gap shrinks** from 0.022 (default) to 0.008–0.014: the tuned models memorise less.
- Smaller learning rates need more trees, so they train slower (`mean_fit_time` up to 6 s vs about 1.5 s for the best).

In [5]:
cv_res = pd.DataFrame(search.cv_results_)

cv_res["gap"] = cv_res["mean_train_score"] - cv_res["mean_test_score"]

COLUMNS = [f"param_{name}"for name in param_distributions] + [
    "mean_test_score",
    "std_test_score",
    "mean_train_score",
    "gap",
    "mean_fit_time",
]

cv_res.sort_values("rank_test_score")[COLUMNS].head(10).round(4)





,param_learning_rate,param_max_leaf_nodes,param_min_samples_leaf,param_l2_regularization,mean_test_score,std_test_score,mean_train_score,gap,mean_fit_time
29,0.0675,21,378,1.1044,0.8663,0.0039,0.8788,0.0125,1.4406
8,0.0575,10,326,0.0063,0.8662,0.0039,0.8737,0.0075,1.8177
10,0.0679,16,365,5.8723,0.8661,0.0039,0.8760,0.0099,1.6774
28,0.0150,21,150,0.0774,0.8661,0.0043,0.8778,0.0117,4.8066
9,0.0463,25,407,0.5263,0.8660,0.0038,0.8798,0.0138,2.1425
4,0.0206,28,180,2.1368,0.8660,0.0039,0.8801,0.0141,4.0839
1,0.0170,26,234,0.2481,0.8660,0.0040,0.8799,0.0139,4.9944
18,0.0302,23,256,0.0015,0.8660,0.0039,0.8794,0.0134,2.8761
21,0.0129,14,28,1.6173,0.8659,0.0040,0.8736,0.0077,5.6252
17,0.0135,30,271,4.8696,0.8658,0.0042,0.8789,0.0130,5.9048


## Step 6 · Best candidate vs default, fold by fold

The search used the same `cv` object as the baseline, so fold *i* contains the same rows on both sides and the per-fold differences are a fair, paired comparison (as in notebook 03, step 6).

Caution: this candidate is the **best of 30**, chosen on these same 5 folds, so its score is slightly optimistic: part of its lead is the luck of being picked.

In [6]:
best = search.best_index_

SPLIT_COLUMNS = [f'split{i}_test_score' for i in range(N_SPLITS)]
tuned = cv_res.loc[best, SPLIT_COLUMNS].to_numpy()

diff = tuned - baseline["test_score"]

per_fold = pd.DataFrame({
    "baseline": baseline["test_score"],
    "tuned": tuned,
    "difference": diff,
})
per_fold.index.name = 'fold'
per_fold.round(4)

,baseline,tuned,difference
fold,,,
0,0.8587,0.8606,0.0019
1,0.8663,0.8663,0.0001
2,0.8682,0.8687,0.0005
3,0.8617,0.8640,0.0023
4,0.8707,0.8720,0.0013


In [7]:
# Decision rule from step 3 (same as notebook 04)
MIN_WINS = 4        # tuned model must win on at least 4 of 5 folds
MIN_GAIN = 0.001    # and raise mean ROC-AUC by at least this much

wins = (diff > 0).sum()
mean_gain = diff.mean()

print(f"wins: {wins} / {N_SPLITS}    mean gain: {mean_gain:+.4f}    worst: {diff.min():+.4f}")

if wins >= MIN_WINS and mean_gain >= MIN_GAIN:
    print("Adopt the tuned hyperparameters")
else:
    print("Keep the default hyperparameters")


wins: 5 / 5    mean gain: +0.0012    worst: +0.0001
Adopt the tuned hyperparameters


## Step 7 · Check the rounded settings saved in `src`

The search returned values like `learning_rate = 0.0674644…`. Digits like these are noise, so readable rounded values were saved in `src/credit_risk_ai/models/pipeline.py` (`TUNED_HGB_PARAMS`, used by `build_tuned_hist_gradient_boosting()`):

`max_iter=500, learning_rate=0.067, max_leaf_nodes=21, min_samples_leaf=378, l2_regularization=1.1`

Rounding changes the model slightly, so it must be measured again with the same folds and the same rule. The gain is printed with 5 decimals, and the rule is applied to the unrounded value.

In [8]:
tuned_cv = cross_validate(build_tuned_hist_gradient_boosting(), X, y, cv=cv,
                          scoring="roc_auc", return_train_score=True, n_jobs=-1)

tuned_val = tuned_cv["test_score"].mean()
tuned_train = tuned_cv["train_score"].mean()
final_diff = tuned_cv["test_score"] - baseline["test_score"]    # paired: same folds as the baseline
final_wins = (final_diff > 0).sum()
final_gain = final_diff.mean()

print(f"validation: {tuned_val:.4f}    (default {val:.4f})")
print(f"gap:        {tuned_train - tuned_val:.4f}    (default {train - val:.4f})")
print(f"fit time:   {tuned_cv['fit_time'].mean():.1f} s     (default {baseline['fit_time'].mean():.1f} s)")
print(f"wins: {final_wins} / {N_SPLITS}    mean gain: {final_gain:+.5f}    worst: {final_diff.min():+.5f}")

# Same rule as step 6, applied to the unrounded gain
if final_wins >= MIN_WINS and final_gain >= MIN_GAIN:
    print("Rule: passes")
else:
    print("Rule: does not pass")

validation: 0.8661    (default 0.8651)
gap:        0.0125    (default 0.0223)
fit time:   1.2 s     (default 0.7 s)
wins: 5 / 5    mean gain: +0.00098    worst: +0.00002
Rule: does not pass


## Step 8 · Conclusions

**Setup:** 30 random hyperparameter combinations for HistGradientBoosting, each scored with the same 5-fold CV as notebooks 03-04 and compared with the default model fold by fold. Decision rule fixed in advance: at least 4/5 folds AND mean gain ≥ +0.001.

| Model | Validation ROC-AUC | Train/validation gap | Gain vs default | Folds improved | Fit time |
|---|---|---|---|---|---|
| Default | 0.8651 | 0.0223 | — | — | 0.7 s |
| Best of 30 (search values) | 0.8663 | 0.0125 | +0.0012 | 5 / 5 | 1.5 s |
| Rounded, saved in `src` | 0.8661 | 0.0125 | **+0.00098** | 5 / 5 | 1.2 s |

**Findings**

1. **Tuning clearly reduces overfitting.** The gap drops from 0.022 to about 0.013. The settings that do this have a small `learning_rate` and mostly a large `min_samples_leaf`; `l2_regularization` barely matters.
2. **The ROC-AUC gain is about +0.001, right at the edge of the rule.** The best of 30 passes (+0.0012), but it was picked on these same folds and is slightly optimistic. Rounding its values to readable numbers gives +0.00098, just below the bar. Changing only the 4th decimal of `learning_rate` moves the result by about ±0.0002 (and can turn a 5/5 into a 4/5), so differences of this size are noise.
3. **Many settings are equally good.** The top 10 candidates are within 0.0005 of each other: a plateau, not one sharp optimum.

**Decision: keep the default hyperparameters.** The rule was fixed before the search so that the result could not be bent afterwards, and the rounded tuned model does not meet it. The tuned settings stay in `src/credit_risk_ai/models/pipeline.py` (`TUNED_HGB_PARAMS`, `build_tuned_hist_gradient_boosting()`) marked as not adopted, for reference.

**What this means:** HistGradientBoosting is close to its ceiling on this data (about 0.865; the best Kaggle scores were about 0.869-0.870). Neither feature engineering (notebook 04) nor tuning moved it by a meaningful amount.

**Next step:** calibration and the decision threshold, using out-of-fold predictions of the default HistGradientBoosting (with `add_features`); then train the final model on all training data and predict the Kaggle test set.